# CatchUp AI — Grounded Lecture Recovery with Gemma Vision

**Build With Gemma @ Bangladesh — Multimodal Track**

Every day, thousands of Bangladeshi university students miss a lecture — traffic, illness, monsoon flooding, or part-time jobs. They borrow a friend's notebook and a photo of the board, but never know what they actually missed.

**CatchUp AI** compares a classroom whiteboard photo against a student's notebook photo and produces a grounded, triaged gap report: what's missing, what's incorrect, and what to study first — ranked by importance, not just listed.

This is not OCR and not a note generator. Gemma performs **semantic comparison** between two sources, using the board as ground truth, so nothing in the output is invented — it's either present on the board or flagged as absent from the notes.

This notebook walks through the full pipeline: setup, the prompt, the live Gemma call, the parsed structured output, and the rendered report.

## 1. Setup

In [ ]:
!pip install -q google-genai pillow

import os
import json
import base64
from io import BytesIO
from PIL import Image
from IPython.display import display, Markdown, HTML

# In Kaggle: Add-ons -> Secrets -> add GEMMA_API_KEY, then attach it to this notebook
try:
    from kaggle_secrets import UserSecretsClient
    API_KEY = UserSecretsClient().get_secret("GEMMA_API_KEY")
except Exception:
    API_KEY = os.environ.get("GEMMA_API_KEY", "")

MODEL = os.environ.get("GEMMA_MODEL", "gemma-3-27b-it")
print("API key loaded:", bool(API_KEY))

## 2. Load the two source images

`board.jpg` — the classroom whiteboard (ground truth of what was taught)

`notes.jpg` — the student's own or borrowed notebook page

Replace these paths with your own sample images uploaded as a Kaggle Dataset attached to this notebook (or upload directly into the working directory).

In [ ]:
BOARD_PATH = "/kaggle/input/catchup-samples/board.jpg"
NOTES_PATH = "/kaggle/input/catchup-samples/notes.jpg"

def to_b64_jpeg(path, max_side=1400):
    img = Image.open(path).convert("RGB")
    img.thumbnail((max_side, max_side))
    buf = BytesIO()
    img.save(buf, format="JPEG", quality=88)
    return base64.b64encode(buf.getvalue()).decode()

display(Image.open(BOARD_PATH))
display(Image.open(NOTES_PATH))

## 3. The Prompt

This is the core design decision in the project. Three things matter here:

1. **Semantic, not literal, comparison** — content counts as covered if the *meaning* was captured, even in shorthand, Bangla, or different words.
2. **Explainable importance scoring** — Gemma rates each gap 1–5 against explicit criteria (deadline/definition/boundary condition vs. a minor example), so the score isn't arbitrary.
3. **A `partially_covered` bucket** — related-but-incomplete content is a genuine third category, not forced into "missing" or "correct".

In [ ]:
PROMPT = """You are analysing two images from a single university lecture in Bangladesh.

IMAGE 1 = the classroom whiteboard/blackboard (the GROUND TRUTH of what was taught).
IMAGE 2 = a student's handwritten notebook page (possibly borrowed, rushed, or incomplete).
Both may mix Bangla and English. Handwriting may be messy.

Your task is SEMANTIC comparison of lecture content, not pixel or exact-string matching.
Content counts as covered if the student captured the MEANING, even in different words,
shorthand, Bangla instead of English, or abbreviated form.

Rate importance 1-5 using ONLY these criteria:
  5 = a deadline, assignment, exam announcement, core definition, theorem, or a
      correctness-critical detail (boundary condition, constraint, edge case)
  4 = a prerequisite concept or a formula the rest of the topic depends on
  3 = a worked method or procedure step
  2 = a supporting illustration or secondary example
  1 = an aside, restatement, or decorative content

Return ONLY valid JSON. No markdown fences, no commentary.

{
  "coverage_percent": <int 0-100, share of board content meaningfully captured>,
  "topic": "<short lecture topic name>",
  "missing_from_notes": [
    {"content": "", "importance": 1-5, "reason": "<why this importance, per criteria>"}
  ],
  "corrections": [
    {"in_notes": "", "on_board": "", "importance": 1-5, "reason": ""}
  ],
  "partially_covered": [
    {"in_notes": "", "on_board": "", "what_to_add": ""}
  ],
  "clean_notes": "<complete, well-structured Markdown notes of the BOARD only. Never invent content that is not visible on the board.>"
}"""

print(PROMPT)

## 4. Call Gemma

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=API_KEY)

response = client.models.generate_content(
    model=MODEL,
    contents=[
        types.Content(role="user", parts=[
            types.Part.from_bytes(data=base64.b64decode(to_b64_jpeg(BOARD_PATH)), mime_type="image/jpeg"),
            types.Part.from_bytes(data=base64.b64decode(to_b64_jpeg(NOTES_PATH)), mime_type="image/jpeg"),
            types.Part.from_text(text=PROMPT),
        ])
    ],
)

raw_text = response.text.strip()
print(raw_text)

## 5. Parse the structured output

In [ ]:
cleaned = raw_text.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
result = json.loads(cleaned)
result

## 6. Render the Gap Report

This is the same triaged view shown in the live Streamlit app — coverage score, critical gaps sorted by importance, corrections, partial matches, and the verified board notes.

In [ ]:
def stars(n):
    return "★" * int(n) + "☆" * (5 - int(n))

cov = result.get("coverage_percent", 0)
display(HTML(f"<h2>📊 Coverage: {cov}%</h2>"))
display(Markdown(f"**Topic:** {result.get('topic', '—')}"))

missing = sorted(result.get("missing_from_notes", []), key=lambda x: -x.get("importance", 0))
display(Markdown(f"### 🔴 Critical Missing ({len(missing)})"))
for m in missing:
    display(Markdown(f"**{stars(m['importance'])} — {m['content']}**  \n_{m.get('reason','')}_"))

corrections = result.get("corrections", [])
if corrections:
    display(Markdown(f"### 🟡 Corrections ({len(corrections)})"))
    for c in corrections:
        display(Markdown(f"Your note: `{c['in_notes']}` → Board: `{c['on_board']}`  \n{stars(c.get('importance',3))} — _{c.get('reason','')}_"))

partial = result.get("partially_covered", [])
if partial:
    display(Markdown(f"### 🟠 Partially Covered ({len(partial)})"))
    for p in partial:
        display(Markdown(f"You wrote: `{p['in_notes']}`  \nBoard had: `{p['on_board']}`  \n→ {p['what_to_add']}"))

display(Markdown("### 🟢 Verified Board Notes"))
display(Markdown("_Every line below is grounded in the board image — nothing is generated from outside it._"))
display(Markdown(result.get("clean_notes", "")))

## 7. How Gemma Is Used

- **Model:** Gemma (vision-capable variant), used as-is via API — no fine-tuning was required because the task is grounded comparison against a provided source image, not open-ended generation.
- **Why Gemma fits:** the model must jointly read two handwritten, mixed Bangla/English images and reason about semantic equivalence between them — a multimodal understanding task, not simple OCR.
- **Prompt design:** the prompt fixes an explicit 1-5 importance rubric so scores are explainable rather than arbitrary, and separates `missing_from_notes` from `partially_covered` so near-matches (e.g. "recursion" vs. "O(log n) recursive") aren't misclassified as either fully missing or fully correct.
- **Grounding:** `clean_notes` is explicitly constrained to content visible on the board, reducing hallucination risk compared to free-form note generation.

## 8. Limitations & Future Work

- Accuracy depends on handwriting legibility in both images; very messy notebook photos reduce confidence.
- Currently a single-student, single-session tool — no persistence across a semester's lectures yet.
- **Future direction:** running the same comparison across an entire class's notebooks could produce teacher-facing completeness analytics (e.g. "85% of students missed the boundary condition"), surfacing common misconceptions without building a second application.